# VPython in Google Colab — demo

Runs the `feat/colab-frontend` branch of vpython-jupyter: the whole VPython
protocol over Colab's Jupyter-comm shim, GlowScript loaded from jsDelivr.
Run the cells **in order**. The 3D scene appears in the "VPython" box
(normally under cell 2 — the frontend self-heals into a later cell if
Colab drops that output).

Not supported in Colab (raise a clear error instead of hanging):
`compound`, `text`, `extrusion`, `scene.pause`/`waitfor`, mouse picking —
they need a mid-cell reply the Colab messaging model cannot deliver.
Everything else — objects, `rate()` animation, zoom/orbit, textures — works.

In [ ]:
# Cell 1: install the branch (~1 min; pure-python wheel, no compilation)
!VPYTHON_PURE_PYTHON=1 pip install -q git+https://github.com/vpython/vpython-jupyter@feat/colab-frontend
print('installed')

In [ ]:
# Cell 2: import. A status box should appear below; the kernel keeps
# retrying the connection from its idle loop until the box acks.
from vpython import *
print('vpython imported (colab frontend)')

In [ ]:
# Cell 3: connection checkpoint — expect connected: True within a couple
# seconds of cell 2's box saying 'ready'. Re-run if False; if there is
# NO box anywhere, run: wc.show()
import time
import vpython.with_colab as wc
time.sleep(2)  # give the idle-loop handshake a beat after the JS loads
print('connected:', wc.sender.connected, '| buffered packages:', wc.sender.pending())

In [ ]:
# Cell 4: build the scene (draws immediately if connected, else flushes
# in a burst the moment the connection lands)
floor = box(pos=vec(0, -1, 0), size=vec(6, 0.2, 6), color=color.green)
ball = sphere(pos=vec(0, 2, 0), radius=0.4, color=color.red, make_trail=True)
ball.v = vec(0.4, 0, 0.2)
g = vec(0, -9.8, 0)
earth = sphere(pos=vec(-2, 1.5, -1), radius=0.6, texture=textures.earth)
print('objects created')

In [ ]:
# Cell 5: animate — ~10 s bouncing ball (rate() self-clocks the flush)
dt = 0.01
for _ in range(1000):
    rate(100)
    ball.v = ball.v + g * dt
    ball.pos = ball.pos + ball.v * dt
    if ball.pos.y - ball.radius < floor.pos.y + 0.1:
        ball.v.y = -ball.v.y * 0.9
print('done — try zoom (scroll) and orbit (right-drag) on the scene')